<a href="https://colab.research.google.com/github/dulibarri777-cpu/financial-ledger-audit/blob/main/Milestone_3_Financial_Reconciliation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# 1. Company Internal General Ledger (What we thought we paid)
ledger_data = {
    'Invoice_ID': ['INV-1001', 'INV-1002', 'INV-1003', 'INV-1004', 'INV-1005', 'INV-1006'],
    'Vendor': ['Amazon AWS', 'Google Cloud', 'WeWork Office', 'Dell Hardware', 'Staples Supplies', 'Salesforce CRM'],
    'Ledger_Amount': [14250.00, 8500.00, 12000.00, 4300.00, 520.00, 6100.00],
    'Issue_Date': ['2026-03-01', '2026-03-03', '2026-03-05', '2026-03-10', '2026-03-12', '2026-03-15']
}
df_ledger = pd.DataFrame(ledger_data)

# 2. Bank Clearance Statement (What actually cleared the bank account)
# Notice: INV-1004 never cleared, and INV-1002 cleared with an incorrect charge!
bank_data = {
    'Invoice_ID': ['INV-1001', 'INV-1002', 'INV-1003', 'INV-1005', 'INV-1006', 'BANK-FEE-99'],
    'Bank_Amount': [14250.00, 8900.00, 12000.00, 520.00, 6100.00, 75.00],
    'Cleared_Date': ['2026-03-02', '2026-03-05', '2026-03-06', '2026-03-14', '2026-03-16', '2026-03-31']
}
df_bank = pd.DataFrame(bank_data)

print("--- 1. COMPANY INTERNAL LEDGER ---")
print(df_ledger)
print("\n--- 2. BANK CLEARANCE STATEMENT ---")
print(df_bank)

--- 1. COMPANY INTERNAL LEDGER ---
  Invoice_ID            Vendor  Ledger_Amount  Issue_Date
0   INV-1001        Amazon AWS        14250.0  2026-03-01
1   INV-1002      Google Cloud         8500.0  2026-03-03
2   INV-1003     WeWork Office        12000.0  2026-03-05
3   INV-1004     Dell Hardware         4300.0  2026-03-10
4   INV-1005  Staples Supplies          520.0  2026-03-12
5   INV-1006    Salesforce CRM         6100.0  2026-03-15

--- 2. BANK CLEARANCE STATEMENT ---
    Invoice_ID  Bank_Amount Cleared_Date
0     INV-1001      14250.0   2026-03-02
1     INV-1002       8900.0   2026-03-05
2     INV-1003      12000.0   2026-03-06
3     INV-1005        520.0   2026-03-14
4     INV-1006       6100.0   2026-03-16
5  BANK-FEE-99         75.0   2026-03-31


In [ ]:
# 1. Merge both systems side-by-side on 'Invoice_ID'
df_recon = pd.merge(df_ledger, df_bank, on='Invoice_ID', how='outer')

# 2. Calculate dollar discrepancy (Variance)
df_recon['Variance'] = df_recon['Bank_Amount'].fillna(0) - df_recon['Ledger_Amount'].fillna(0)

# 3. Create an automated audit status flag
def audit_flag(row):
    if pd.isna(row['Bank_Amount']):
        return "CRITICAL: Uncleared / Missing Payment"
    elif pd.isna(row['Ledger_Amount']):
        return "ALERT: Unrecorded Bank Charge"
    elif row['Variance'] != 0:
        return f"VARIANCE: Discrepancy (${row['Variance']:,.2f})"
    else:
        return "CLEARED: Match"

df_recon['Audit_Status'] = df_recon.apply(audit_flag, axis=1)

# 4. Display the Executive Reconciliation Report
print("=== AUTOMATED FINANCIAL RECONCILIATION REPORT ===")
cols = ['Invoice_ID', 'Vendor', 'Ledger_Amount', 'Bank_Amount', 'Variance', 'Audit_Status']
print(df_recon[cols].to_string(index=False))

=== AUTOMATED FINANCIAL RECONCILIATION REPORT ===
 Invoice_ID           Vendor  Ledger_Amount  Bank_Amount  Variance                          Audit_Status
BANK-FEE-99              NaN            NaN         75.0      75.0         ALERT: Unrecorded Bank Charge
   INV-1001       Amazon AWS        14250.0      14250.0       0.0                        CLEARED: Match
   INV-1002     Google Cloud         8500.0       8900.0     400.0       VARIANCE: Discrepancy ($400.00)
   INV-1003    WeWork Office        12000.0      12000.0       0.0                        CLEARED: Match
   INV-1004    Dell Hardware         4300.0          NaN   -4300.0 CRITICAL: Uncleared / Missing Payment
   INV-1005 Staples Supplies          520.0        520.0       0.0                        CLEARED: Match
   INV-1006   Salesforce CRM         6100.0       6100.0       0.0                        CLEARED: Match
